In [3]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, TensorDataset
import os
import os.path as osp
from PIL import Image, ImageFont, ImageDraw
import pandas as pd
import numpy as np
import sys
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
import re
from math import ceil
from matplotlib.cm import get_cmap

In [5]:

input_file = "C:/Users/rrai/dinov2/WSI/Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.tif"
tile_size = 128  # Example tile size

 # Load the image
Image.MAX_IMAGE_PIXELS = None  # Disable decompression bomb error for large images
image = Image.open(input_file)

# Get original dimensions
orig_width, orig_height = image.size
b    # Calculate new dimensions divisible by tile_size
new_width = (orig_width // tile_size) * tile_size
new_height = (orig_height // tile_size) * tile_size

    # Calculate cropping box (center crop)
crop_left = (orig_width - new_width) // 2
crop_top = (orig_height - new_height) // 2
crop_right = orig_width - crop_left - new_width
crop_bottom = orig_height - crop_top - new_height

# Perform the crop
print("Performing cropping. Make take a while for large files")
crop_pic = image.crop((crop_left, crop_top, orig_width - crop_right, orig_height - crop_bottom))
crop_pic.size




Performing cropping. Make take a while for large files


(27520, 20480)

In [ ]:
def create_image_tiles(input_file, output_folder, tile_size=256, file = crop_pic):
    """
    Create equal-sized image tiles from a cropped image and save them as PNG files with coordinate-based filenames.
    
    Parameters:
        input_file (str): Path to the input cropped image file.
        output_folder (str): Path to the folder where the tiles will be saved.
        tile_size (int): Size of the square tiles (default is 64x64).
    """
    # Open the cropped image
    #image = Image.open(input_file)
    Image.MAX_IMAGE_PIXELS = None
    image = file
    img_width, img_height = image.size

    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Generate unique identifier (based on input filename without extension)
    unique_id = os.path.splitext(os.path.basename(input_file))[0]

    # Calculate total number of tiles
    total_tiles = (img_width // tile_size) * (img_height // tile_size)

    # Progress bar setup
    print(f"Generating tiles for image '{input_file}'...")
    with tqdm(total=total_tiles, desc="Processing Tiles", unit="tile") as pbar:
        tile_count = 0
        for y in range(0, img_height, tile_size):
            for x in range(0, img_width, tile_size):
                # Crop the tile
                tile = image.crop((x, y, x + tile_size, y + tile_size))
                
                # Construct the filename: {unique_id}_{x}_{y}.png
                tile_filename = f"{unique_id}_{x}_{y}.png"
                tile_path = os.path.join(output_folder, tile_filename)
                
                # Save the tile
                tile.save(tile_path, format="PNG")
                
                tile_count += 1
                pbar.update(1)  # Update progress bar

    print(f"Created {tile_count} tiles in '{output_folder}'.")


# use
output_folder = "cropped_breast_128"     # Folder to store the tiles
create_image_tiles(input_file, output_folder,tile_size=64, file=crop_pic)

In [14]:
print(f"cropped image: {asizeof.asizeof(crop_pic)} bytes")

cropped image: 1832 bytes


In [9]:
os.chdir("C:/Users/rrai/dinov2")
print("Current Working Directory:", os.getcwd())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Current Working Directory: C:\Users\rrai\dinov2


device(type='cuda')

In [ ]:
DINO_PATH_FINETUNED_DOWNLOADED='C:/Users/rrai/dinov2/pretrained/dinov2_vits_NCT_10k_training_1999_teacher_checkpoint.PTH'

def get_dino_finetuned_downloaded():
    # load the original DINOv2 model with the correct architecture and parameters. The positional embedding is too large.
    # load vits or vitg
    model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    # load finetuned weights
    pretrained = torch.load(DINO_PATH_FINETUNED_DOWNLOADED, map_location=torch.device('cuda'))
    # make correct state dict for loading
    new_state_dict = {}
    for key, value in pretrained['teacher'].items():
        if 'dino_head' in key:
            print('not used')
        else:
            new_key = key.replace('backbone.', '')
            new_state_dict[new_key] = value
    #change shape of pos_embed, shape depending on vits or vitg
    pos_embed = nn.Parameter(torch.zeros(1, 257, 384))
    #pos_embed = nn.Parameter(torch.zeros(1, 257, 1536))
    model.pos_embed = pos_embed
    # load state dict
    model.load_state_dict(new_state_dict, strict=True)
    return model

model=get_dino_finetuned_downloaded()
model.to(device)
model.eval()
print("Model loaded successfully!")

In [ ]:
import psutil
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetUtilizationRates, nvmlDeviceGetMemoryInfo


#for extrating features from unlabeled test images
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define ImageDataset to dynamically load images
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform if transform else transforms.ToTensor()  # Default to ToTensor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

# img feature extraction function

def extract_features_batchwise(loader, model, device):
    """
    Extract features dynamically batch-by-batch while monitoring system performance.

    Args:
        loader (DataLoader): DataLoader for images.
        model (torch.nn.Module): Model for feature extraction.
        device (torch.device): Device to run the model on.

    Returns:
        np.ndarray: Extracted features for all images in the loader.
    """
    all_features = []
    print("Extracting features from images...")

    # Initialize performance monitoring
    cpu_usage = []
    gpu_usage = []
    ram_usage_mb = []
    gpu_mem_usage = []

    process = psutil.Process()

    start_time = time.time()  # Start timing

    # Initialize NVML for GPU monitoring
    if torch.cuda.is_available():
        nvmlInit()
        handle = nvmlDeviceGetHandleByIndex(0)  # Assuming a single GPU

    with torch.no_grad():  # Disable gradient calculation for inference
        for batch_idx, images in enumerate(tqdm(loader, desc="Processing batches")):
            # Record CPU & RAM usage before processing
            cpu_usage.append(psutil.cpu_percent())
            ram_usage_mb.append(process.memory_info().rss / (1024 ** 2))  # Convert to MB

            images = images.to(device)
            outputs = model(images)

            # Record GPU utilization & memory usage
            if torch.cuda.is_available():
                gpu_util = nvmlDeviceGetUtilizationRates(handle).gpu
                gpu_memory = torch.cuda.memory_allocated(device) / (1024 ** 2)  # Convert to MB
                gpu_usage.append(gpu_util)
                gpu_mem_usage.append(gpu_memory)

            all_features.append(outputs.cpu().numpy())  # Move outputs to CPU and convert to NumPy

            # Free memory
            del images, outputs
            torch.cuda.empty_cache()


    # Compute min, max, and average for CPU, GPU, RAM
    cpu_min, cpu_max, cpu_avg = min(cpu_usage), max(cpu_usage), sum(cpu_usage) / len(cpu_usage)
    gpu_min, gpu_max, gpu_avg = (min(gpu_usage), max(gpu_usage), sum(gpu_usage) / len(gpu_usage)) if gpu_usage else (0, 0, 0)
    ram_min, ram_max, ram_avg = min(ram_usage_mb), max(ram_usage_mb), sum(ram_usage_mb) / len(ram_usage_mb)
    gpu_mem_min, gpu_mem_max, gpu_mem_avg = (min(gpu_mem_usage), max(gpu_mem_usage), sum(gpu_mem_usage) / len(gpu_mem_usage)) if gpu_mem_usage else (0, 0, 0)

    # Print performance metrics
   
    print(f"CPU Usage: Min {cpu_min:.2f}%, Max {cpu_max:.2f}%, Avg {cpu_avg:.2f}%")
    print(f"RAM Usage: Min {ram_min:.2f} MB, Max {ram_max:.2f} MB, Avg {ram_avg:.2f} MB")
    if torch.cuda.is_available():
        print(f"GPU Utilization: Min {gpu_min:.2f}%, Max {gpu_max:.2f}%, Avg {gpu_avg:.2f}%")
        print(f"GPU Memory: Min {gpu_mem_min:.2f} MB, Max {gpu_mem_max:.2f} MB, Avg {gpu_mem_avg:.2f} MB")

    all_features = np.vstack(all_features)  # Combine all features into a single array
    end_time = time.time()  # End timing

    # Calculate total time
    total_time = end_time - start_time
    print(f"\nFeature extraction complete. Total time taken: {total_time:.2f} seconds.")
    return all_features

#usage
if __name__ == "__main__":
    # Paths to all images
    dataroot = 'C:/Users/rrai/dinov2/WSI/cropped_breast_128'
    all_image_paths = [os.path.join(dataroot, img) for img in os.listdir(dataroot) if img.endswith(('.png', '.jpg', '.jpeg', '.tif'))]
    test_image_paths = [os.path.join(dataroot, img) for img in os.listdir(dataroot) if img.endswith(('.png', '.jpg', '.jpeg', '.tif'))]
    #LOAD!!
    dataset = ImageDataset(all_image_paths, transform=transform)
    loader = DataLoader(dataset, batch_size=512, num_workers=0, shuffle=False)  #num_workers not working??


    # Extract features for all images
    all_features = extract_features_batchwise(loader, model, device)
    print(f"Extracted features shape: {all_features.shape}")



Extracting features from images...


Processing batches: 100%|██████████████████████████████████████████████████████████████| 68/68 [02:16<00:00,  2.01s/it]

CPU Usage: Min 5.10%, Max 10.40%, Avg 8.32%
RAM Usage: Min 6708.92 MB, Max 6875.92 MB, Avg 6832.19 MB
GPU Utilization: Min 0.00%, Max 48.00%, Avg 2.59%
GPU Memory: Min 540.63 MB, Max 577.38 MB, Avg 576.84 MB

Feature extraction complete. Total time taken: 136.78 seconds.
Extracted features shape: (34776, 384)


In [30]:
print(f"loader: {asizeof.asizeof(loader)} bytes")
print(f"dataset: {asizeof.asizeof(dataset)} bytes")
print(f"all_image_paths: {asizeof.asizeof(all_image_paths)} bytes")

loader: 13058440 bytes
dataset: 13056416 bytes
all_image_paths: 13044384 bytes


In [13]:
from pympler import asizeof
print(f"all_features: {asizeof.asizeof(all_features)} bytes")

all_features: 53416080 bytes
